### Passo 1: Instalar Python 3.10 e dependências
Execute os comandos abaixo para instalar uma versão compatível com o PyCaret.

In [1]:
# --- Célula Extra 2: AutoML com PyCaret ---
# 1. Garantir que o pip existe para o Python 3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

# 2. Instalar dependências ignorando pacotes bloqueados do sistema (como blinker)
!python3.10 -m pip install -q --ignore-installed numpy==1.24.3 pandas pycaret

# Nota: Após a instalação, você deve garantir que o código seja executado usando o executável python3.10
# ou reiniciar o ambiente para que o kernel reconheça as mudanças.

In [ ]:
# ---Setup com XGBoost ---
# Reiniciar Spark com o pacote do XGBoost
try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession

spark = SparkSession.builder \
.master("local[*]") \
.appName("Lab02_XGBoost") \
.config("spark.jars.packages", "ml.dmlc:xgboost4j-spark_2.12:1.7.6") \
.getOrCreate()

print("Spark com XGBoost configurado!")
# (Passo 2: Gerando Dados de Churn) Vamos simular um dataset de telecomunicações.

Spark com XGBoost configurado!


In [ ]:
# ---  Gerando Dados Fake ---
from pyspark.sql.functions import rand, when, col

# Criar 100 mil clientes
df = spark.range(0, 100000).select(
(rand(seed=42) * 100).alias("fatura_mensal"),
(rand(seed=23) * 10).alias("anos_contrato"),
(rand(seed=11) * 50).alias("idade"),
(rand(seed=99)).alias("score_satisfacao")
)

# Criar Label (Churn) com base em regras (para o modelo aprender algo)
# Se satisfação baixa e fatura alta -> Churn (1)
df = df.withColumn("churn",
when((col("score_satisfacao") < 0.3) & (col("fatura_mensal") > 80), 1.0)
.otherwise(0.0)
)

print("Distribuição do Target:")
df.groupBy("churn").count().show()
# (Passo 3: Pipeline com Random Forest (Nativo)) Primeiro, vamos usar o Random Forest nativo do Spark.


Distribuição do Target:
+-----+-----+
|churn|count|
+-----+-----+
|  0.0|93998|
|  1.0| 6002|
+-----+-----+



In [ ]:
from pycaret.classification import *
import pandas as pd

In [ ]:
# Converter Spark DataFrame para Pandas (apenas para demo)
from pyspark.sql.functions import rand

# Criar 100 mil clientes
df = spark.range(0, 100000).select(
    (rand(seed=42) * 100).alias("fatura_mensal"),
    (rand(seed=23) * 10).alias("anos_contrato"),
    (rand(seed=11) * 50).alias("idade"),
    (rand(seed=99)).alias("score_satisfacao")
)

# Simular a coluna churn baseada no seu código anterior
from pyspark.sql.functions import when, col
df = df.withColumn("churn", when((col("score_satisfacao") < 0.3) & (col("fatura_mensal") > 80), 1.0).otherwise(0.0))

# Em produção, você faria sampling primeiro
df_pandas = df.limit(5000).toPandas()

# Setup do PyCaret (removido o argumento 'silent' que causava o erro)
clf_setup = setup(data=df_pandas, target='churn', session_id=42, verbose=False)

# Comparar TODOS os modelos disponíveis
print("Comparando 15+ modelos automaticamente...")
best_models = compare_models(n_select=3)

# O PyCaret treinou os modelos e retornou os 3 melhores
print("\nTop 3 Modelos:")
for i, model in enumerate(best_models):
  print(f"{i+1}. {type(model).__name__}")

# Pegar o melhor
best = best_models[0]

# Fazer predições
predictions = predict_model(best, data=df_pandas)
# No PyCaret 3.x, o nome da coluna de predição é 'prediction_label'
print(predictions[['churn', 'prediction_label']].head())

Comparando 15+ modelos automaticamente...


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
xgboost,Extreme Gradient Boosting,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0510
dt,Decision Tree Classifier,0.9997,0.9976,0.9952,1.0000,0.9976,0.9974,0.9974,0.0300
rf,Random Forest Classifier,0.9997,1.0000,0.9952,1.0000,0.9976,0.9974,0.9974,0.2910
gbc,Gradient Boosting Classifier,0.9997,0.9973,0.9952,1.0000,0.9976,0.9974,0.9974,0.4080
ada,Ada Boost Classifier,0.9994,0.9979,0.9902,1.0000,0.9950,0.9947,0.9948,0.2060
lightgbm,Light Gradient Boosting Machine,0.9994,0.9998,1.0000,0.9909,0.9953,0.9950,0.9951,0.1980
et,Extra Trees Classifier,0.9986,1.0000,0.9857,0.9905,0.9880,0.9873,0.9873,0.2000
qda,Quadratic Discriminant Analysis,0.9946,0.9993,0.9324,0.9761,0.9529,0.9501,0.9508,0.0260
nb,Naive Bayes,0.9929,0.9995,0.8890,0.9902,0.9359,0.9321,0.9342,0.0260
lr,Logistic Regression,0.9811,0.9953,0.7743,0.8982,0.8293,0.8194,0.8232,0.0660


Processing:   0%|          | 0/67 [00:00<?, ?it/s]


Top 3 Modelos:
1. XGBClassifier
2. DecisionTreeClassifier
3. RandomForestClassifier


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.9988,1.0000,0.9865,0.9932,0.9899,0.9892,0.9892


   churn  prediction_label
0    0.0                 0
1    0.0                 0
2    0.0                 0
3    0.0                 0
4    0.0                 0


### Entendendo a Saída do PyCaret

O que acabou de acontecer no código acima foi um processo de **AutoML (Machine Learning Automatizado)**. Aqui está como interpretar os resultados:

1.  **`setup()`**: Esta função preparou os dados. Ela identificou automaticamente quais colunas são números, quais são categorias e tratou o alvo (`churn`).
2.  **`compare_models()`**: O PyCaret testou mais de 15 algoritmos diferentes (como Random Forest, XGBoost, Regressão Logística) usando uma técnica chamada *Cross-Validation*.
    *   A tabela que aparece mostra métricas como **Accuracy (Acurácia)**, **AUC** e **F1-Score**.
    *   **Dica:** O modelo no topo da lista é aquele que teve o melhor desempenho médio nos testes.
3.  **`predict_model()`**: Aplicamos o melhor modelo encontrado aos dados para ver o que ele diria.

#### Como ler a tabela de predições final:
Ao olhar as últimas linhas impressas, você verá duas colunas principais:
*   **`churn`**: É o valor real (o que aconteceu de verdade).
*   **`prediction_label`**: É o que o modelo previu.
*   **`prediction_score`** (se visível): É a confiança do modelo (ex: 0.95 significa que ele tem 95% de certeza).

**Exercício:** Se o `churn` for 1.0 e o `prediction_label` for 1.0, o modelo acertou! Se forem diferentes, o modelo cometeu um erro de classificação.

> **Atenção:** Após mudar a versão do Python, você provavelmente precisará reinstalar o PyCaret e o Pandas usando `!python3.10 -m pip install pycaret pandas` para garantir que fiquem vinculados à versão correta.